# ASTRA AION-ARC Ω — Raccordement Kaggle sans mot de passe dans le chat

**Objectif** : exécuter le candidat v2.9 dans un environnement Python cloud gratuit, avec Kaggle CLI, puis lancer la **véritable soumission de compétition** (réexécution cachée).

Le notebook utilise d'abord une autorisation Kaggle déjà présente, notamment le secret privé Colab `KAGGLE_API_TOKEN`. Si elle n'existe pas, Kaggle propose sa connexion OAuth officielle. **Ce lien doit être approuvé par le titulaire du compte** ; aucune ancienne session Kaggle ne peut être reproduite à partir d'une simple connexion Google, et aucun code ne contourne l'authentification.

Ce notebook ne contient ni token, ni mot de passe, ni appel à un service payant. La seule infrastructure GPU requise pour l'évaluation est celle de Kaggle, sous réserve de son quota.

La validation locale de Kaggle (phase A) et le score officiel après réexécution cachée (phase B) sont distingués dans le script. Échéance : **2 novembre 2026, 23:59 UTC**.
## Audit sans soumission
La dernière cellule est désormais strictement en lecture seule : validation de l'autorisation `drstarck`, lecture des soumissions officielles déjà présentes, sans relancer l'ancien candidat ni dépenser le quota. La soumission reste une étape indépendante après ce contrôle.


In [ ]:
import os, sys, subprocess, pathlib, shutil
ROOT = pathlib.Path("/content/AION-ARC")
BRANCH = "astra/arc3-emergency-20261031"
REMOTE = "https://github.com/Dr-starck66/AION-ARC.git"
def astra_call(argv, *, cwd=None, check=True):
    p = subprocess.run(argv, cwd=cwd, check=check)
    return p.returncode
print("ASTRA_COLAB_SETUP_START", flush=True)
astra_call([sys.executable, "-m", "pip", "install", "-q", "kaggle>=2.2.4"])
if not ROOT.exists():
    astra_call(["git", "clone", "--quiet", "--depth", "1", "--single-branch", "--branch", BRANCH, REMOTE, str(ROOT)])
else:
    astra_call(["git", "fetch", "--quiet", "--depth", "1", "origin", BRANCH], cwd=ROOT)
    astra_call(["git", "checkout", "--quiet", "-B", BRANCH, "FETCH_HEAD"], cwd=ROOT)
assert (ROOT/"kaggle/v29-milestone2/submission.ipynb").is_file()
assert (ROOT/"scripts/arc3_kaggle_final_run.py").is_file()
print("ASTRA_COLAB_SETUP_PASS", flush=True)


In [ ]:
# Reuse the account owner's saved Colab secret when it exists. It is never printed.
import os, subprocess, shutil
try:
    from google.colab import userdata
    saved = userdata.get("KAGGLE_API_TOKEN")
    if saved:
        os.environ["KAGGLE_API_TOKEN"] = saved
        print("ASTRA_AUTH_SOURCE: PRIVATE_COLAB_SECRET", flush=True)
    del saved
except Exception:
    print("ASTRA_AUTH_SOURCE: NO_ACCESSIBLE_COLAB_SECRET", flush=True)
if not shutil.which("kaggle"):
    raise RuntimeError("Kaggle CLI unavailable")
probe = subprocess.run(["kaggle", "kernels", "status", "drstarck/notebook2700e63354"],
                       text=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if probe.returncode != 0:
    print("ASTRA_OAUTH_REQUIRED: official Kaggle authorization link follows. Approve it in the browser while this cell is running.", flush=True)
    # DO NOT use --force; preserve preexisting credentials if the CLI detects them.
    subprocess.run(["kaggle", "auth", "login", "--no-launch-browser"], check=True)
    probe = subprocess.run(["kaggle", "kernels", "status", "drstarck/notebook2700e63354"],
                           text=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if probe.returncode != 0:
    raise RuntimeError("Kaggle auth not verified for owner drstarck; no upload or submission")
print("ASTRA_PRIVATE_KAGGLE_AUTH: PASS (drstarck historic notebook)", flush=True)


In [ ]:
# Read-only: prove Kaggle CLI can access the user's private notebook and
# read the *current official* submission history. No GPU run, push or submit.
import sys, subprocess, json
assert (ROOT / "scripts/arc3_kaggle_final_run.py").is_file()
p = subprocess.run([sys.executable, "scripts/arc3_kaggle_final_run.py", "--check-auth"],
                   cwd=ROOT)
if p.returncode != 0:
    raise RuntimeError("Kaggle account connection not verified. No upload or submission attempted.")
receipt = ROOT / "reports/arc3-final-20261101/receipt.json"
if receipt.is_file():
    data = json.loads(receipt.read_text())
    summary = {k: data[k] for k in
               ("private_kernel_auth", "auth_probe", "submissions_read",
                "recent_submissions", "at_utc") if k in data}
    print("ASTRA_KAGGLE_AUTH_RECEIPT", json.dumps(summary, ensure_ascii=False))
print("ASTRA_COLAB_AUTH_AUDIT_PASS; no changes made to Kaggle", flush=True)
